# 6 — FastAPI benchmark: sequential and concurrent
Runs in the **Jupyter container on node-serve-system**.
The FastAPI service runs in the `fastapi` container on the same Docker Compose network.

In [ ]:
import time, json
import numpy as np
import requests
import concurrent.futures

BASE_URL = "http://fastapi:8000"

In [ ]:
# Health check
print(requests.get(f"{BASE_URL}/health").json())

In [ ]:
def make_payload(n_songs=10):
    return {
        "user_features": np.random.rand(32).tolist(),
        "candidate_songs": [
            {"song_id": f"song_{i}", "features": np.random.rand(32).tolist()}
            for i in range(n_songs)
        ],
    }

In [ ]:
# Sequential benchmark — 100 requests
N = 100
latencies = []
for _ in range(N):
    t0 = time.time()
    r = requests.post(f"{BASE_URL}/rank", json=make_payload())
    latencies.append(time.time() - t0)
    assert r.status_code == 200, r.text

print(f"Sequential ({N} requests):")
print(f"  p50: {np.percentile(latencies, 50)*1000:.2f} ms")
print(f"  p95: {np.percentile(latencies, 95)*1000:.2f} ms")
print(f"  p99: {np.percentile(latencies, 99)*1000:.2f} ms")
print(f"  Throughput: {N/sum(latencies):.2f} req/sec")

In [ ]:
# Concurrent benchmark — 1000 requests, 16 workers
N_WORKERS  = 16
N_REQUESTS = 1000

def send_one(_):
    t0 = time.time()
    r = requests.post(f"{BASE_URL}/rank", json=make_payload())
    return time.time() - t0, r.status_code

t_start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    results = list(pool.map(send_one, range(N_REQUESTS)))
t_wall = time.time() - t_start

lats   = [r[0] for r in results]
errors = sum(1 for r in results if r[1] != 200)

print(f"Concurrent ({N_REQUESTS} requests, {N_WORKERS} workers):")
print(f"  p50: {np.percentile(lats, 50)*1000:.2f} ms")
print(f"  p95: {np.percentile(lats, 95)*1000:.2f} ms")
print(f"  p99: {np.percentile(lats, 99)*1000:.2f} ms")
print(f"  Throughput: {N_REQUESTS/t_wall:.2f} req/sec")
print(f"  Errors: {errors} ({errors/N_REQUESTS*100:.1f}%)")

In [ ]:
# Smoke test
payload = {
    "user_features": np.random.rand(32).tolist(),
    "candidate_songs": [
        {"song_id": "track_jazz_001",   "features": np.random.rand(32).tolist()},
        {"song_id": "track_pop_042",    "features": np.random.rand(32).tolist()},
        {"song_id": "track_metal_007",  "features": np.random.rand(32).tolist()},
        {"song_id": "track_folk_013",   "features": np.random.rand(32).tolist()},
        {"song_id": "track_classic_99", "features": np.random.rand(32).tolist()},
    ],
}
r = requests.post(f"{BASE_URL}/rank", json=payload)
for entry in r.json():
    print(f"  {entry['song_id']:25s}  score={entry['score']:.4f}")

with open("smoke_output.json", "w") as f:
    json.dump(r.json(), f, indent=2)